[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cafzal/frontier/blob/main/examples/cuopt_bidirectional/cuopt_bidirectional.ipynb)

# Frontier × cuOpt — bidirectional benefits, across three problems

**The question.** Does wrapping NVIDIA **cuOpt** inside Frontier's multi-objective engine help
*both* tools? We test it honestly on three problems — a **portfolio** (allocate, QP), a
**supplier network** (allocate + per-region caps, QP), and a **capital-project portfolio**
(select, MILP) — by running each **three ways**:

- **cuOpt alone** — cuOpt is single-objective, so you commit to one weighting and get **one
  plan**, blind to what you traded away.
- **Frontier alone (NSGA-II)** — an *approximate* multi-objective frontier, no exact guarantees.
- **Frontier + cuOpt** — the frontier, **exact at each point**, with cuOpt's optimality/duals.

**What to look for (the honest bidirectional read):**
- **Frontier → cuOpt (large, structural):** the lone star vs the frontier. cuOpt alone gives one
  point; Frontier turns it into an explorable tradeoff surface — a capability cuOpt lacks.
- **cuOpt → Frontier (narrow):** exact corners + duals (see the portfolio dual at the end). On
  *coverage*, NSGA already matches the integration — cuOpt adds certificates, not more points.

**Caveat, stated up front:** these datasets are **synthetic**. The Frontier→cuOpt benefit is
structural (realism-independent); the cuOpt→cuOpt coverage parity may partly be a "well-conditioned
random data" artifact — we don't claim cuOpt beats NSGA on coverage.

**Requirements.** cuOpt needs **Linux + an NVIDIA GPU** — pick a Colab GPU runtime
(*Runtime → Change runtime type → T4 GPU*) and *Run all*. Engine + data clone from the public repo.

In [ ]:
import subprocess
try:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                         capture_output=True, text=True)
    print(out.stdout.strip() or "no nvidia-smi output")
except FileNotFoundError:
    print("No NVIDIA GPU detected - cuOpt cannot run. In Colab: Runtime -> Change runtime type -> GPU (T4).")

In [ ]:
# Bootstrap: clone Frontier (public) + install cuOpt + engine deps. First run ~2-3 min.
import os, subprocess, sys
REPO = "/content/frontier"
if not os.path.isdir(REPO):
    rc = subprocess.run(["git", "clone", "-q", "https://github.com/cafzal/frontier.git", REPO],
                        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"}).returncode
    if rc != 0 or not os.path.isdir(REPO):
        raise RuntimeError("Could not clone github.com/cafzal/frontier - check the connection, then re-run.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "--extra-index-url", "https://pypi.nvidia.com", "cuopt-cu12"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pymoo>=0.6", "pydantic>=2.0", "scipy>=1.11", "pandas>=2.0", "matplotlib>=3.7"], check=True)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print("Frontier on path:", REPO in sys.path)

In [ ]:
import json, os
import numpy as np
import matplotlib.pyplot as plt
from engine.models import Problem
from engine.optimizer import optimize
from solvers import cuopt_backend as cb
# Confirms the GPU solver is live (fails if not on a cuOpt runtime):
from cuopt.linear_programming.problem import Problem as CuProblem
print("cuOpt import OK - GPU solver live.")

In [ ]:
REPO = "/content/frontier"

def load_example(name):
    p = json.load(open(f"{REPO}/examples/{name}/problem.json"))
    s = json.load(open(f"{REPO}/examples/{name}/scores.json"))
    return Problem(**{**p, **s})

def _points(run, objs):
    return np.array([[s.objective_values[o.name] for o in objs] for s in run.solutions])

def cuopt_alone(problem):
    """What cuOpt gives WITHOUT Frontier: ONE exact point for one scalarization
    (optimize the first objective), ignoring the multi-objective tradeoff."""
    objs = problem.objectives
    if problem.approach.value == "binary":
        n, names, S, dirs, mc = cb._build_milp_data(problem)
        sel, ok = cb._solve_milp_cuopt(dirs[0] * S[:, 0], [], mc, n)
        return {o.name: float(S[:, j] @ sel) for j, o in enumerate(objs)}
    score = cb._opt._build_score_matrix(problem)
    im = cb._opt._build_interaction_matrices(problem)
    ri, _ = cb._resolve_objective_roles(problem)
    cov = cb._nearest_psd(im[ri])
    cp = cb._opt._parse_constraints(problem)
    mw = (cp["max_allocation"] / 100.0) if cp.get("max_allocation") else None
    lin = cb._resolve_linear_objectives(problem)
    mu = score[:, lin[0]]
    groups = cb._group_limits(problem)
    support = None
    if groups:                      # group-feasible support so the single solve is valid
        maximize = objs[lin[0]].direction.value == "maximize"
        support = np.array([i for grp, gmax in groups
                            for i in sorted(grp, key=lambda i: mu[i], reverse=maximize)[:gmax]])
    w, ok = cb._solve_qp_cuopt(cov, mu, None, True, mw, support=support)
    prop = cb._opt._ProportionalProblem(n_options=len(problem.options), score_matrix=score,
                                        objectives=objs, interaction_matrices=im, **cp)
    W = (w * 100).reshape(1, -1)
    return {o.name: float(prop._aggregate_objective(W, j)[0]) for j, o in enumerate(objs)}

def three_way(name, ex, xi, yi):
    problem = load_example(ex)
    objs = problem.objectives
    os.environ.pop("FRONTIER_SOLVER", None)
    nsga = _points(optimize(problem, seed=42), objs)
    os.environ["FRONTIER_SOLVER"] = "cuopt"
    integ = _points(optimize(problem, seed=42), objs)
    a = cuopt_alone(problem)
    ax_, ay_ = objs[xi].name, objs[yi].name
    print(f"{name}: cuOpt-alone = 1 point | Frontier alone (NSGA) = {len(nsga)} pts | "
          f"Frontier + cuOpt = {len(integ)} pts")
    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    ax.scatter(nsga[:, xi], nsga[:, yi], s=42, color="darkorange", alpha=0.6,
               label=f"Frontier alone / NSGA ({len(nsga)})")
    ax.scatter(integ[:, xi], integ[:, yi], s=46, facecolor="none", edgecolor="navy",
               linewidth=1.4, zorder=3, label=f"Frontier + cuOpt ({len(integ)})")
    ax.scatter([a[ax_]], [a[ay_]], s=260, marker="*", color="crimson", edgecolor="k",
               zorder=5, label="cuOpt alone (1 point)")
    ax.set_xlabel(ax_); ax.set_ylabel(ay_)
    ax.set_title(f"{name}: cuOpt alone is ONE point - Frontier makes it a frontier")
    ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
    return problem

## 1 - Portfolio (allocate across 30 ETFs - QP)

Return / volatility (quadratic) / yield. cuOpt alone returns the single minimum-risk portfolio;
Frontier turns its exact QP into the whole risk/return frontier.

In [ ]:
three_way("Portfolio", "investment_portfolio", 0, 1)

## 2 - Supplier network (allocate order share across 25 suppliers - QP + per-region caps)

cost / reliability / lead-time / ESG / quadratic concentration-risk, at most 3 suppliers per
region. cuOpt's QP respects the per-region caps via group-aware support selection.

In [ ]:
three_way("Supplier network", "supplier_selection", 0, 1)

## 3 - Capital projects (select a subset - MILP)

NPV / cost / risk / strategic-fit under a hard budget + dependencies + exclusions. cuOpt alone
returns the single max-NPV plan; Frontier maps the whole tradeoff via the exact MILP per point.

In [ ]:
three_way("Capital projects", "capital_project_selection", 0, 1)

## 4 - cuOpt -> Frontier: the narrow benefit (exact duals)

Coverage washes (NSGA matches the integration above), but cuOpt brings something NSGA cannot: exact
**duals**. On the portfolio, the dual of the return constraint is the *shadow price of return* - the
marginal risk you accept for one more unit of return - rising monotonically along the efficient edge.

In [ ]:
problem = load_example("investment_portfolio")
score = cb._opt._build_score_matrix(problem); im = cb._opt._build_interaction_matrices(problem)
ri, _ = cb._resolve_objective_roles(problem); lin = cb._resolve_linear_objectives(problem)
cov = cb._nearest_psd(im[ri]); mu = score[:, lin[0]]
cp = cb._opt._parse_constraints(problem); mw = (cp["max_allocation"]/100.0) if cp.get("max_allocation") else None
w_mv, _ = cb._solve_qp_cuopt(cov, mu, None, True, mw)
r_lo, r_hi = float(mu @ w_mv), float(mu.max())
rets, duals = [], []
for tgt in np.linspace(r_lo, r_hi * 0.999, 25):
    n = len(mu); p = CuProblem("dual_demo")
    wv = [p.addVariable(lb=0.0, ub=mw if mw else 1.0, name=f"w{i}") for i in range(n)]
    quad = None
    for i in range(n):
        for j in range(n):
            c = float(cov[i, j])
            if abs(c) > 1e-12:
                t = c * wv[i] * wv[j]; quad = t if quad is None else quad + t
    from cuopt.linear_programming.problem import MINIMIZE
    p.setObjective(quad, sense=MINIMIZE)
    p.addConstraint(sum(wv) == 1, name="budget")
    rc = p.addConstraint(sum(float(mu[i]) * wv[i] for i in range(n)) >= float(tgt), name="return_target")
    p.solve()
    if getattr(p.Status, "name", "") not in ("Optimal", "PrimalFeasible"):
        continue
    rets.append(float(mu @ np.array([wv[i].Value for i in range(n)]))); duals.append(abs(float(rc.DualValue)))
fig, ax = plt.subplots(); ax.plot(rets, duals, "o-", color="purple", lw=1.5)
ax.set_xlabel("Expected Return achieved (%)"); ax.set_ylabel("Shadow price d(variance)/d(return)")
ax.set_title("cuOpt dual - the marginal risk cost of return (NSGA cannot produce this)")
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
print(f"shadow price {min(duals):.3f} -> {max(duals):.3f} as return rises - exact, from cuOpt's dual.")

## 5 - Conclusions: the integration is bidirectional (and honest)

Across all three problems, one picture repeats:

- **Frontier -> cuOpt (the large, structural benefit).** The red star is everything cuOpt gives you
  *alone*: **one plan**, for one weighting you had to guess, with no view of the tradeoff. Frontier
  turns cuOpt's exact solve into an **explorable frontier** - the multi-objective decision layer a
  single-objective solver structurally lacks. This holds on every problem and is realism-independent.
- **cuOpt -> Frontier (the narrow benefit).** The navy (Frontier+cuOpt) and orange (NSGA) frontiers
  **overlap** - on *coverage*, NSGA already matches the integration. What cuOpt adds is **exactness**:
  every point is provably optimal for its scalarization, plus **duals** (4) NSGA cannot produce. On
  these well-conditioned synthetic problems that is a narrow edge; we do **not** claim cuOpt beats
  NSGA on coverage.

**Most compelling for the cuOpt team:** the **capital-projects MILP** and the **supplier network** -
both land cuOpt on the kind of exact, hard-constrained subproblem it is built for, while Frontier
supplies the multi-objective exploration cuOpt's goal-programming front end does not.

**One-line thesis:** *cuOpt makes each point exact; Frontier makes the set a decision.* The frontier
itself - not just one optimal point - is what the integration adds, and exactness + duals are what
cuOpt adds back.